# Vanguard TLP Data Analysis Case Study

## Step 1: Extract

##### Import Pandas, Numpy, and SQLite3 into python file

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy import stats
import statsmodels.formula.api as smf

##### Exploratory Data Analysis on "demo.xlsm"

In [2]:
demo_raw = "demo.xlsm"
df_demo_raw = pd.read_excel(demo_raw, engine='openpyxl')

In [3]:
df_demo_raw.head()

,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,campaign,click_dt,bounce_dt,open_dt
0,1054161513,6,11,1,70377.57,6.0,44.2,0,0,NaT,NaT,NaT
1,5761909860,7,13,2,63505.69,6.6,44.2,0,1,NaT,NaT,NaT
2,9130086586,6,13,2,55933.94,5.9,43.7,1,1,NaT,NaT,NaT
3,5496433327,6,12,2,81072.63,5.5,45.5,0,1,NaT,NaT,NaT
4,8917877079,8,12,0,87534.41,3.8,45.3,1,0,NaT,NaT,2020-12-14


In [4]:
df_demo_raw.tail()

,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,campaign,click_dt,bounce_dt,open_dt
41762,139438696,7,14,1,73981.75,5.4,47.0,0,2,NaT,NaT,NaT
41763,8080126468,7,12,0,83325.66,7.4,46.1,0,2,2021-01-06,NaT,2020-11-26
41764,9805524809,7,15,0,46040.38,3.2,47.2,0,2,NaT,NaT,NaT
41765,767386300,5,12,0,60608.12,6.7,45.7,0,2,NaT,NaT,NaT
41766,9047256786,7,12,0,47131.32,6.2,44.1,0,2,2021-02-13,NaT,2021-01-15


In [5]:
df_demo_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41767 entries, 0 to 41766
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   part_id    41767 non-null  int64         
 1   fund_ct    41767 non-null  int64         
 2   logon_ct   41767 non-null  int64         
 3   call_ct    41767 non-null  int64         
 4   balance    41767 non-null  float64       
 5   tenure     41767 non-null  float64       
 6   age        41767 non-null  float64       
 7   gender     41767 non-null  int64         
 8   campaign   41767 non-null  int64         
 9   click_dt   10507 non-null  datetime64[ns]
 10  bounce_dt  31 non-null     datetime64[ns]
 11  open_dt    21020 non-null  datetime64[ns]
dtypes: datetime64[ns](3), float64(3), int64(6)
memory usage: 3.8 MB


In [6]:
df_demo_raw.describe(include='all')

,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,campaign,click_dt,bounce_dt,open_dt
count,4.176700e+04,41767.000000,41767.000000,41767.000000,41767.000000,41767.000000,41767.000000,41767.000000,41767.000000,10507,31,21020
mean,4.999553e+09,6.291474,12.041301,1.147221,60046.147449,6.007113,44.999588,0.272703,0.735916,2020-12-22 06:09:29.448938752,2020-08-15 12:54:09.677419520,2020-10-30 11:49:39.333967616
min,1.332140e+05,5.000000,10.000000,0.000000,-7984.650000,1.700000,42.700000,0.000000,0.000000,2020-07-30 00:00:00,2020-07-30 05:30:00,2020-07-30 00:00:00
25%,2.509040e+09,6.000000,11.000000,0.000000,49879.165000,5.300000,44.300000,0.000000,0.000000,2020-11-25 00:00:00,2020-07-30 05:30:00,2020-09-14 00:00:00
50%,5.010379e+09,6.000000,12.000000,2.000000,60043.100000,6.000000,44.800000,0.000000,1.000000,2021-01-04 00:00:00,2020-08-23 07:46:00,2020-10-31 00:00:00
75%,7.506106e+09,7.000000,13.000000,2.000000,70151.315000,6.700000,45.700000,1.000000,1.000000,2021-01-27 00:00:00,2020-08-25 07:56:00,2020-12-15 00:00:00
max,9.999918e+09,12.000000,19.000000,5.000000,126301.310000,10.000000,48.900000,2.000000,2.000000,2021-02-13 00:00:00,2020-08-29 07:20:00,2021-01-31 00:00:00
std,2.890176e+09,0.752298,1.091707,1.053628,15031.505233,1.001498,0.914382,0.445409,0.639023,NaN,NaN,NaN


In [7]:
df_demo_raw.isna().sum()

part_id          0
fund_ct          0
logon_ct         0
call_ct          0
balance          0
tenure           0
age              0
gender           0
campaign         0
click_dt     31260
bounce_dt    41736
open_dt      20747
dtype: int64

In [8]:
df_demo_raw.duplicated().sum()

np.int64(0)

In [9]:
df_demo_raw['part_id'].nunique()

41767

In [10]:
df_demo_raw['campaign'].value_counts()

campaign
1    21799
0    15499
2     4469
Name: count, dtype: int64

In [11]:
df_demo_raw[['click_dt','open_dt','bounce_dt']].notna().sum()

click_dt     10507
open_dt      21020
bounce_dt       31
dtype: int64

In [12]:
df_demo_raw.groupby('campaign').agg(open_rate=('open_dt', 
    lambda x: x.notna().mean()), click_rate=('click_dt', 
    lambda x: x.notna().mean()))

,open_rate,click_rate
campaign,,
0,0.503129,0.248016
1,0.505023,0.254324
2,0.495189,0.250392


In [13]:
df_demo_raw["click_dt"].between('2020-07-30','2020-08-29')

0        False
1        False
2        False
3        False
4        False
         ...  
41762    False
41763    False
41764    False
41765    False
41766    False
Name: click_dt, Length: 41767, dtype: bool

In [14]:
df_demo_raw[(df_demo_raw['bounce_dt'].notna()) & (df_demo_raw['open_dt'].notna())]

,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,campaign,click_dt,bounce_dt,open_dt
3187,5076654141,6,11,0,71891.56,5.7,44.4,1,0,NaT,2020-07-30 05:30:00,2020-08-29
3342,9576372771,7,12,2,76234.12,8.0,44.0,0,1,2021-02-09,2020-07-30 05:30:00,2021-01-13
3762,9428035314,6,11,0,50163.65,7.7,45.1,0,0,2021-01-14,2020-07-30 05:30:00,2020-10-20
4475,6483195957,5,12,2,50509.41,4.7,45.7,0,1,2021-01-25,2020-07-30 05:30:00,2020-09-22
6988,3781719974,7,13,2,59719.95,6.7,43.7,0,1,NaT,2020-08-29 07:20:00,2020-08-12
8773,8890838825,6,13,0,53322.22,7.2,44.0,0,0,2021-01-08,2020-07-30 05:30:00,2020-11-12
11895,929930576,5,12,0,51369.13,5.8,45.9,0,0,2020-12-05,2020-08-25 07:55:00,2020-09-19
22561,4470791647,6,13,2,50951.18,5.0,46.5,0,1,2020-12-31,2020-08-25 07:48:00,2020-08-15
22745,8119335137,8,12,2,95954.96,6.3,45.4,0,1,2020-12-20,2020-08-29 07:20:00,2020-11-26
23140,8400298473,6,11,2,42767.90,7.0,44.4,0,1,NaT,2020-08-29 07:19:00,2020-11-29


##### Exploratory Data Analysis on "enroll_data.xlsm"

In [15]:
enroll_raw = "enroll_data.xlsm"
df_enroll_raw = pd.read_excel(enroll_raw, engine='openpyxl')


In [16]:
df_enroll_raw.head()

,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt
0,8652971234,1,0,Enrolled,2020-08-09,NaT
1,6966233343,1,0,Enrolled,2020-12-08,NaT
2,6621762485,1,0,Enrolled,2020-07-15,NaT
3,2079270798,1,0,Enrolled,2020-11-15,NaT
4,4826079770,1,0,Enrolled,2020-08-20,NaT


In [17]:
df_enroll_raw.tail()

,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt
159,514690838,1,0,Enrolled,2020-07-28,NaT
160,8029667255,1,0,Enrolled,2020-10-06,NaT
161,4657603855,0,0,No Longer In Plan,2020-12-15,NaT
162,3039087731,1,0,Enrolled,2020-09-29,NaT
163,6481540801,1,0,Enrolled,2020-08-03,NaT


In [18]:
df_enroll_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   part_id            164 non-null    int64         
 1   curr_enrll_fl      164 non-null    int64         
 2   unenrll_fl         164 non-null    int64         
 3   curr_enrll_status  164 non-null    object        
 4   enrll_dt           164 non-null    datetime64[ns]
 5   unenrll_dt         8 non-null      datetime64[ns]
dtypes: datetime64[ns](2), int64(3), object(1)
memory usage: 7.8+ KB


In [19]:
df_enroll_raw.describe(include='all')

,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt
count,1.640000e+02,164.000000,164.000000,164,164,8
unique,NaN,NaN,NaN,3,NaN,NaN
top,NaN,NaN,NaN,Enrolled,NaN,NaN
freq,NaN,NaN,NaN,132,NaN,NaN
mean,5.252179e+09,0.804878,0.048780,NaN,2020-08-18 17:24:52.682926848,2020-08-18 00:00:00
min,1.556957e+08,0.000000,0.000000,NaN,2019-12-20 00:00:00,2020-03-26 00:00:00
25%,2.947491e+09,1.000000,0.000000,NaN,2020-07-15 00:00:00,2020-07-07 18:00:00
50%,5.507128e+09,1.000000,0.000000,NaN,2020-08-13 00:00:00,2020-07-20 12:00:00
75%,7.659814e+09,1.000000,0.000000,NaN,2020-09-29 00:00:00,2020-11-03 00:00:00
max,9.953890e+09,1.000000,1.000000,NaN,2020-12-17 00:00:00,2020-12-17 00:00:00


In [20]:
df_enroll_raw.isna().sum()

part_id                0
curr_enrll_fl          0
unenrll_fl             0
curr_enrll_status      0
enrll_dt               0
unenrll_dt           156
dtype: int64

In [21]:
df_enroll_raw.duplicated().sum()

np.int64(21)

In [22]:
df_enroll_raw['part_id'].nunique()

143

In [23]:
df_enroll_raw['curr_enrll_status'].value_counts()

curr_enrll_status
Enrolled                   132
No Longer In Plan           24
Unenrolled From Service      8
Name: count, dtype: int64

In [24]:
df_enroll_raw[df_enroll_raw['enrll_dt'].between('2020-07-30', '2020-08-29')]

,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt
0,8652971234,1,0,Enrolled,2020-08-09,NaT
4,4826079770,1,0,Enrolled,2020-08-20,NaT
12,3717113197,1,0,Enrolled,2020-08-26,NaT
29,208054155,0,0,No Longer In Plan,2020-08-22,NaT
30,877088406,1,0,Enrolled,2020-08-25,NaT
35,1284743023,1,0,Enrolled,2020-08-26,NaT
55,6111536911,1,0,Enrolled,2020-08-03,NaT
58,9695467488,1,0,Enrolled,2020-08-20,NaT
59,9672869463,0,0,No Longer In Plan,2020-08-07,NaT
64,2671956384,1,0,Enrolled,2020-08-14,NaT


In [25]:
(df_enroll_raw['unenrll_dt'] - df_enroll_raw['enrll_dt']).dt.days.describe()

count      8.000000
mean      86.750000
std      135.470345
min        0.000000
25%        0.000000
50%       10.500000
75%      122.500000
max      363.000000
dtype: float64

In [26]:
df_enroll_raw[df_enroll_raw['part_id'].duplicated(keep=False)].sort_values('part_id')

,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt
132,805806679,0,0,No Longer In Plan,2020-06-30,NaT
37,805806679,0,0,No Longer In Plan,2020-06-30,NaT
104,1786057426,1,0,Enrolled,2020-09-01,NaT
87,1786057426,1,0,Enrolled,2020-09-01,NaT
119,1842774690,0,0,No Longer In Plan,2020-06-30,NaT
46,1842774690,0,0,No Longer In Plan,2020-06-30,NaT
43,1869231292,1,0,Enrolled,2020-09-01,NaT
97,1869231292,1,0,Enrolled,2020-09-01,NaT
36,1926202122,1,0,Enrolled,2020-09-23,NaT
85,1926202122,1,0,Enrolled,2020-09-23,NaT


In [27]:
(df_enroll_raw['unenrll_dt'] < df_enroll_raw['enrll_dt']).sum()

np.int64(0)

##### Exploratory Data Analysis on "web_data.xlsx"

In [28]:
web_raw = "web_data.xlsx"
df_web_raw = pd.read_excel(web_raw, engine='openpyxl')

In [29]:
df_web_raw.head()

,part_id,web_session_id,hit_dt,page
0,6442286637,9993505343,2021-01-31,welcome
1,6442286637,9993505343,2021-01-31,financial-profile
2,6442286637,9993505343,2021-01-31,financial-profile
3,6442286637,9993505343,2021-01-31,financial-profile
4,6442286637,9993505343,2021-01-31,financial-profile


In [30]:
df_web_raw.tail()

,part_id,web_session_id,hit_dt,page
77299,9047256786,2906700267,2021-02-13,retirement
77300,9047256786,2906700267,2021-02-13,retirement
77301,9047256786,2906700267,2021-02-13,retirement
77302,9047256786,2906700267,2021-02-13,retirement
77303,9047256786,2906700267,2021-02-13,retirement


In [31]:
df_web_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77304 entries, 0 to 77303
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   part_id         77304 non-null  int64         
 1   web_session_id  77304 non-null  int64         
 2   hit_dt          77304 non-null  datetime64[ns]
 3   page            76963 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 2.4+ MB


In [32]:
df_web_raw.describe(include='all')

,part_id,web_session_id,hit_dt,page
count,7.730400e+04,7.730400e+04,77304,76963
unique,NaN,NaN,NaN,15
top,NaN,NaN,NaN,dashboard
freq,NaN,NaN,NaN,24498
mean,5.016591e+09,4.946523e+09,2020-10-28 16:47:36.752561408,NaN
min,3.053283e+06,5.236090e+05,2020-07-30 00:00:00,NaN
25%,2.711084e+09,2.471043e+09,2020-09-10 00:00:00,NaN
50%,5.095046e+09,4.955351e+09,2020-10-27 00:00:00,NaN
75%,7.420700e+09,7.429216e+09,2020-12-12 00:00:00,NaN
max,9.997506e+09,9.998308e+09,2021-02-13 00:00:00,NaN


In [33]:
df_web_raw.isna().sum()

part_id             0
web_session_id      0
hit_dt              0
page              341
dtype: int64

In [34]:
df_web_raw.duplicated().sum()

np.int64(54789)

In [35]:
df_web_raw['part_id'].nunique()

5027

In [36]:
df_web_raw['web_session_id'].nunique()

13563

In [37]:
df_web_raw[df_web_raw['hit_dt'].between('2020-07-30', '2020-08-29')]

,part_id,web_session_id,hit_dt,page
34,5912079776,5408241737,2020-08-14,welcome
35,5912079776,5408241737,2020-08-14,welcome
36,5912079776,5408241737,2020-08-14,welcome
106,3027826580,8412503084,2020-08-12,welcome
107,3027826580,598774514,2020-08-17,welcome
...,...,...,...,...
77192,7229258722,4518927929,2020-08-11,welcome
77234,6202893508,8379280182,2020-08-27,welcome
77235,6202893508,8379280182,2020-08-27,welcome
77251,139438696,9026288330,2020-08-25,welcome


In [38]:
df_web_raw['page'].value_counts()

page
dashboard             24498
welcome               15002
retirement            14095
financial-profile      8544
risk-attitude          6107
money-to-work          4938
edit-retirement        2617
edit-risk-attitude      252
registration            251
get-cash                212
revisit-retirement      177
debt                    159
pending-offboard         51
unenroll-feedback        50
spending-strategy        10
Name: count, dtype: int64

In [39]:
df_web_raw.groupby('part_id')['page'].apply(lambda x: 'welcome' in x.values)

part_id
3053283       True
7343427       True
8607030       True
9904745       True
11353887      True
              ... 
9987599236    True
9990361441    True
9994167885    True
9996403077    True
9997506498    True
Name: page, Length: 5027, dtype: bool

#### Confirm files integrate together

In [40]:
def check_integration(df_left, df_right, key='part_id'):
    merged = df_left.merge(df_right, on=key, how='outer', indicator=True)
    return merged['_merge'].value_counts()

In [41]:
check_integration(df_demo_raw, df_enroll_raw)

_merge
left_only     41624
both            164
right_only        0
Name: count, dtype: int64

In [42]:
check_integration(df_demo_raw, df_web_raw)

_merge
both          77304
left_only     36740
right_only        0
Name: count, dtype: int64

In [43]:
check_integration(df_web_raw, df_enroll_raw)

_merge
left_only     70644
both           9488
right_only       98
Name: count, dtype: int64

## Transform & Load

- **LOAD** — where data is written into a table in `case_study_clean.db`
- **TRANSFORM** — where the cleaning logic runs (either before a table is created, or as `UPDATE`s against a table already loaded)

Wach dataset is first loaded as-is into a raw table, then cleaned in place with `UPDATE` statements, rather than being fully transformed in memory before any write happens.

### Step 1: Load raw extracts into a SQLite database

Each raw DataFrame from the Extract step is loaded into its own table in a SQLite database. Cleaning is then done with SQL against these tables, and each cleaned result is written back out to pandas and saved as its own table in the database.

In [44]:
conn = sqlite3.connect('case_study_clean.db')

# LOAD: raw extracts written into the database exactly as read, no cleaning yet

df_demo_raw.to_sql('demo_raw', conn, if_exists='replace', index=False)
df_enroll_raw.to_sql('enroll_raw', conn, if_exists='replace', index=False)
df_web_raw.to_sql('web_raw', conn, if_exists='replace', index=False)

# Experiment measurement window (emails were sent 7/30/2020 - 8/29/2020)

WINDOW_START = '2020-07-30 00:00:00'
WINDOW_END   = '2020-08-29 23:59:59'

print('Raw tables loaded:', conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'").fetchall())

Raw tables loaded: [('advisor_funnel',), ('advisor_demographics_funnel',), ('demo_clean',), ('enroll_clean',), ('web_clean',), ('advisor_web_summary',), ('advisor_demographics_web',), ('demo_raw',), ('enroll_raw',), ('web_raw',)]


### Step 2: Clean `demo` → `demo_clean`

Issues found during EDA and how each is handled:
- **Duplicates** — no exact-row or `part_id` duplicates existed, but a defensive `ROW_NUMBER()` dedup (keep first occurrence per `part_id`) is applied in case new extracts introduce any.
- **Invalid values** — 3 advisors have a negative `balance` (AUM can't be negative); set to `NULL` rather than guessing a sign-flip.
- **Inconsistent engagement data** — `campaign = 2` (did not receive an email) but 1,119 rows still show a `click_dt` and 2,213 show an `open_dt`; these are cleared since an advisor who wasn't emailed can't click or open one. Rows with a `bounce_dt` (19 also show `open_dt`, 10 also show `click_dt`) are treated as never-delivered, so `click_dt`/`open_dt` are cleared. Two rows have a `click_dt` with no `open_dt` — an advisor can't click a link without the email being opened, so `open_dt` is backfilled from `click_dt`.
- **Date window** — `click_dt`, `open_dt`, and `bounce_dt` are only meaningful inside the 7/30/2020–8/29/2020 send window; values outside it are cleared.
- **Text standardization** — `gender` and `campaign` are numeric codes with no free text to trim/case-fold, so readable `gender_label` / `campaign_label` columns are added alongside the original codes.

In [45]:
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS demo_clean")

# TRANSFORM + LOAD: dedup, add readable label columns, and write the results table in one statement

cur.execute("""
    CREATE TABLE demo_clean AS
    SELECT part_id, fund_ct, logon_ct, call_ct, balance, tenure, age,
           gender,
           CASE gender WHEN 0 THEN 'Male' WHEN 1 THEN 'Female' WHEN 2 THEN 'Unknown' END AS gender_label,
           campaign,
           CASE campaign WHEN 0 THEN 'Email A' WHEN 1 THEN 'Email B' WHEN 2 THEN 'No Email' END AS campaign_label,
           click_dt, bounce_dt, open_dt
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY part_id ORDER BY rowid) AS rn
        FROM demo_raw
    ) t
    WHERE rn = 1
""")

# TRANSFORM (in place): clean up invalid/inconsistent values now that

cur.execute("UPDATE demo_clean SET balance = NULL WHERE balance < 0")
cur.execute("UPDATE demo_clean SET click_dt = NULL, bounce_dt = NULL, open_dt = NULL WHERE campaign = 2")
cur.execute("UPDATE demo_clean SET click_dt = NULL, open_dt = NULL WHERE bounce_dt IS NOT NULL")
cur.execute("UPDATE demo_clean SET open_dt = click_dt WHERE click_dt IS NOT NULL AND open_dt IS NULL")

# Keep engagement timestamps within the experiment window

for col in ('click_dt', 'bounce_dt', 'open_dt'):
    cur.execute(
        f"UPDATE demo_clean SET {col} = NULL "
        f"WHERE {col} IS NOT NULL AND {col} NOT BETWEEN ? AND ?",
        (WINDOW_START, WINDOW_END)
    )
conn.commit()

# Read the loaded, transformed table back into pandas for downstream use/verification

df_demo_clean = pd.read_sql(
    "SELECT * FROM demo_clean", conn, parse_dates=['click_dt', 'bounce_dt', 'open_dt']
)
df_demo_clean.shape

(41767, 14)

Quick check that the demo cleaning behaved as expected:

In [46]:
print('Rows raw -> clean:', df_demo_raw.shape[0], '->', df_demo_clean.shape[0])
print('Duplicate part_ids remaining:', df_demo_clean['part_id'].duplicated().sum())
print('Negative balances remaining:', (df_demo_clean['balance'] < 0).sum())
print('campaign=2 rows still showing engagement dates:',
      df_demo_clean.loc[df_demo_clean['campaign'] == 2, ['click_dt', 'open_dt', 'bounce_dt']].notna().sum().sum())
print('bounce_dt + open_dt both present:',
      ((df_demo_clean['bounce_dt'].notna()) & (df_demo_clean['open_dt'].notna())).sum())
df_demo_clean.head()

Rows raw -> clean: 41767 -> 41767
Duplicate part_ids remaining: 0
Negative balances remaining: 0
campaign=2 rows still showing engagement dates: 0
bounce_dt + open_dt both present: 0


,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,gender_label,campaign,campaign_label,click_dt,bounce_dt,open_dt
0,133214,8,11,2,85903.15,5.0,45.1,0,Male,1,Email B,NaT,NaT,NaT
1,550202,7,12,2,43173.88,3.3,43.9,0,Male,1,Email B,NaT,NaT,NaT
2,577736,6,10,0,43850.56,8.7,45.1,0,Male,0,Email A,NaT,NaT,NaT
3,619810,5,13,2,80008.50,7.0,46.3,1,Female,1,Email B,NaT,NaT,NaT
4,809111,5,12,2,71166.60,6.6,45.8,0,Male,1,Email B,NaT,NaT,2020-08-10


### Step 3: Clean `enroll_data` → `enroll_clean`

Issues found during EDA and how each is handled:
- **Duplicates** — 21 fully duplicated rows exist; `SELECT DISTINCT` removes them.
- **Text standardization** — `curr_enrll_status` values are already consistently cased, but `TRIM()` is applied defensively.
- **Inconsistent dates** — no rows currently have `unenrll_dt` earlier than `enrll_dt`, but a defensive check clears `unenrll_dt` if that logic error ever appears.
- **Nulls** — `unenrll_dt` is `NULL` for the 156 advisors who never unenrolled; this is expected and left as-is rather than imputed.
- **Date window** — enrollment is a *downstream* outcome that can legitimately occur after the email window (e.g. an advisor who saw the 7/30 email but didn't finish enrolling until October). Deleting or blanking those dates would erase real outcomes the experiment is trying to measure. Instead of forcing `enrll_dt` into the window, an `enrolled_in_window` flag is added so the enrollment data can be sliced either way.


In [47]:
cur.execute("DROP TABLE IF EXISTS enroll_clean")

# TRANSFORM + LOAD: dedup, trim text, add the enrolled_in_window flag, and write the result as a new table in one statement

cur.execute("""
    CREATE TABLE enroll_clean AS
    SELECT DISTINCT
           part_id, curr_enrll_fl, unenrll_fl,
           TRIM(curr_enrll_status) AS curr_enrll_status,
           enrll_dt, unenrll_dt,
           CASE WHEN enrll_dt BETWEEN ? AND ? THEN 1 ELSE 0 END AS enrolled_in_window
    FROM enroll_raw
""", (WINDOW_START, WINDOW_END))

# TRANSFORM (in place): defensive fix now that enroll_clean is already loaded

cur.execute("UPDATE enroll_clean SET unenrll_dt = NULL WHERE unenrll_dt IS NOT NULL AND unenrll_dt < enrll_dt")
conn.commit()

# Read the loaded, transformed table back into pandas for downstream use/verification

df_enroll_clean = pd.read_sql(
    "SELECT * FROM enroll_clean", conn, parse_dates=['enrll_dt', 'unenrll_dt']
)
df_enroll_clean.shape

(143, 7)

Quick check that the enrollment cleaning behaved as expected:

In [48]:
print('Rows raw -> clean:', df_enroll_raw.shape[0], '->', df_enroll_clean.shape[0])
print('Duplicate rows remaining:', df_enroll_clean.duplicated().sum())
print('Duplicate part_ids remaining:', df_enroll_clean['part_id'].duplicated().sum())
print('unenrll_dt before enrll_dt:', (df_enroll_clean['unenrll_dt'] < df_enroll_clean['enrll_dt']).sum())
print(df_enroll_clean['enrolled_in_window'].value_counts())
df_enroll_clean.head()

Rows raw -> clean: 164 -> 143
Duplicate rows remaining: 0
Duplicate part_ids remaining: 0
unenrll_dt before enrll_dt: 0
enrolled_in_window
0    117
1     26
Name: count, dtype: int64


,part_id,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt,enrolled_in_window
0,8652971234,1,0,Enrolled,2020-08-09,NaT,1
1,6966233343,1,0,Enrolled,2020-12-08,NaT,0
2,6621762485,1,0,Enrolled,2020-07-15,NaT,0
3,2079270798,1,0,Enrolled,2020-11-15,NaT,0
4,4826079770,1,0,Enrolled,2020-08-20,NaT,1


### Step 4: Clean `web_data` → `web_clean`

Issues found during EDA and how each is handled:
- **Duplicates** — 69,660 of 77,304 rows (90%) are exact duplicate hits (same `part_id`, `web_session_id`, `hit_dt`, `page`); `SELECT DISTINCT` collapses these to single events.
- **Nulls** — 341 rows have a missing `page`; since `page` is the field the downstream analysis depends on, these rows carry no usable information and are dropped.
- **Text standardization** — `page` values are trimmed and lower-cased for consistency (they were already clean, but this protects against future extracts).
- **Date window** — `hit_dt` is filtered to the 7/30/2020–8/29/2020 experiment window; a large share of the raw hits fall outside it (e.g. into 2021) and aren't part of the A/B test period.


In [49]:
cur.execute("DROP TABLE IF EXISTS web_clean")

# TRANSFORM + LOAD: dedup, drop unusable rows, standardize text, filter to the experiment window, and write the result as a new table in one statement

cur.execute("""
    CREATE TABLE web_clean AS
    SELECT DISTINCT part_id, web_session_id, hit_dt, TRIM(LOWER(page)) AS page
    FROM web_raw
    WHERE page IS NOT NULL
      AND hit_dt BETWEEN ? AND ?
""", (WINDOW_START, WINDOW_END))
conn.commit()

# Read the loaded, transformed table back into pandas for downstream use/verification

df_web_clean = pd.read_sql("SELECT * FROM web_clean", conn, parse_dates=['hit_dt'])
df_web_clean.shape

(3510, 4)

Quick check that the web-hit cleaning behaved as expected:

In [50]:
print('Rows raw -> clean:', df_web_raw.shape[0], '->', df_web_clean.shape[0])
print('Duplicate rows remaining:', df_web_clean.duplicated().sum())
print('Null pages remaining:', df_web_clean['page'].isna().sum())
print('hit_dt range:', df_web_clean['hit_dt'].min(), '->', df_web_clean['hit_dt'].max())
df_web_clean.head()

Rows raw -> clean: 77304 -> 3510
Duplicate rows remaining: 0
Null pages remaining: 0
hit_dt range: 2020-07-30 00:00:00 -> 2020-08-29 00:00:00


,part_id,web_session_id,hit_dt,page
0,5912079776,5408241737,2020-08-14,welcome
1,3027826580,8412503084,2020-08-12,welcome
2,3027826580,598774514,2020-08-17,welcome
3,3027826580,4359460348,2020-08-25,welcome
4,5563701395,7796313433,2020-08-12,welcome


### Step 5: Load confirmation — each cleaned dataset in its own table

`demo_clean`, `enroll_clean`, and `web_clean` were each loaded as their own table in `case_study_clean.db` in the cells above, so the cleaned data is persisted independently of the raw tables and can be reloaded by later notebook sections without re-running any of the transform logic.

In [51]:
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
for (t,) in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f'{t:<15} {n:>7} rows')

advisor_demographics_funnel   41767 rows
advisor_demographics_web   41767 rows
advisor_funnel    41767 rows
advisor_web_summary   41767 rows
demo_clean        41767 rows
demo_raw          41767 rows
enroll_clean        143 rows
enroll_raw          164 rows
web_clean          3510 rows
web_raw           77304 rows


## Analyze

This section answers the two questions the business asked for: whether the creative an advisor received affected (1) accessing the DI sign-up process and (2) enrolling in DI.

The work is split by what each tool is actually good at:
- **SQL** builds a reusable, per-advisor web-activity summary table, so it's a natural continuation of the Transform & Load pattern above, and it's written into `case_study_clean.db` so later work doesn't have to re-derive it.
- **Python** (`scipy`/`statsmodels`) runs the actual hypothesis tests. The z-statistic, p-value, and confidence interval have to be computed in Python.


### Step 6: Build a per-advisor web-activity summary table

One row per advisor summarizing their web activity — every distinct page they hit, total hits and sessions, and whether they reached `welcome` (the page where the DI sign-up process starts) — joined with their enrollment outcome. This feeds directly into the business questions in Step 7.


In [52]:
cur.execute("DROP TABLE IF EXISTS advisor_web_summary")

# TRANSFORM + LOAD: aggregate web hits and enrollment outcome to one row per advisor, and write the result as a new (reusable) table

cur.execute("""
    CREATE TABLE advisor_web_summary AS
    SELECT
        d.part_id,
        d.campaign,
        d.campaign_label,
        GROUP_CONCAT(DISTINCT w.page)                                  AS pages_visited,
        COUNT(w.page)                                                  AS total_web_hits,
        COUNT(DISTINCT w.page)                                         AS distinct_pages_visited,
        COUNT(DISTINCT w.web_session_id)                               AS web_sessions,
        MIN(w.hit_dt)                                                  AS first_hit_dt,
        MAX(w.hit_dt)                                                  AS last_hit_dt,
        MAX(CASE WHEN w.page = 'welcome' THEN 1 ELSE 0 END)            AS reached_welcome,
        MAX(CASE WHEN e.part_id IS NOT NULL THEN 1 ELSE 0 END)         AS ever_enrolled,
        MAX(CASE WHEN e.enrolled_in_window = 1 THEN 1 ELSE 0 END)      AS enrolled_in_window
    FROM demo_clean d
    LEFT JOIN web_clean w ON w.part_id = d.part_id
    LEFT JOIN enroll_clean e ON e.part_id = d.part_id
    GROUP BY d.part_id, d.campaign, d.campaign_label
""")
conn.commit()

df_advisor_web = pd.read_sql(
    "SELECT * FROM advisor_web_summary", conn,
    parse_dates=['first_hit_dt', 'last_hit_dt']
)
df_advisor_web.shape


(41767, 12)

Quick look at advisors and web activity by campaign arm, also worth noting the two email groups aren't an exact 50/50 split.

In [53]:
print("Advisors per campaign arm:")
print(df_advisor_web['campaign_label'].value_counts())
print()
print("Web activity and outcomes by campaign arm:")
df_advisor_web.groupby('campaign_label')[['reached_welcome', 'ever_enrolled', 'total_web_hits', 'distinct_pages_visited']].sum()


Advisors per campaign arm:
campaign_label
Email B     21799
Email A     15499
No Email     4469
Name: count, dtype: int64

Web activity and outcomes by campaign arm:


,reached_welcome,ever_enrolled,total_web_hits,distinct_pages_visited
campaign_label,,,,
Email A,65,8,175,105
Email B,92,28,350,224
No Email,888,107,2985,2104


### Step 7: A/B test — does the creative affect the outcome?

Restricted to advisors who actually received an email (`campaign` 0 or 1); the `No Email` group is excluded since it wasn't part of the randomized assignment. Two-proportion z-tests compare Email A vs. Email B on:
1. Accessed the sign-up process, reached the `welcome` page
2. Enrolled in DI within the measurement period, `enrolled_in_window`


In [54]:
ab = df_advisor_web[df_advisor_web['campaign'].isin([0, 1])]
email_a = ab[ab['campaign'] == 0]
email_b = ab[ab['campaign'] == 1]

metrics = [
    ('reached_welcome', 'Accessed the DI sign-up process'),
    ('enrolled_in_window', 'Enrolled in DI'),
]

results = []
for col, label in metrics:
    count = [email_a[col].sum(), email_b[col].sum()]
    nobs = [len(email_a), len(email_b)]
    stat, pval = proportions_ztest(count, nobs)
    ci_a = proportion_confint(count[0], nobs[0], method='wilson')
    ci_b = proportion_confint(count[1], nobs[1], method='wilson')
    results.append({
        'metric': label,
        'email_a_rate': count[0] / nobs[0],
        'email_a_95ci': ci_a,
        'email_b_rate': count[1] / nobs[1],
        'email_b_95ci': ci_b,
        'z_stat': stat,
        'p_value': pval,
    })

df_ab_results = pd.DataFrame(results)
df_ab_results


,metric,email_a_rate,email_a_95ci,email_b_rate,email_b_95ci,z_stat,p_value
0,Accessed the DI sign-up process,0.004194,"(0.0032920168504067416, 0.005341333311639971)",0.004220,"(0.003442810675454411, 0.005172647047130633)",-0.039042,0.968857
1,Enrolled in DI,0.000065,"(1.1389506559084984e-05, 0.00036540972964612675)",0.000413,"(0.00021722985391965915, 0.0007845413538353652)",-2.025047,0.042863


Reading the results:

- Access to the sign-up process: Email A and Email B perform essentially identically (~0.42% each reach `welcome`); the z-test is not significant (p ≈ 0.97). The creative doesn't change whether an advisor clicks through.

- Enrollment: Email B's enrollment rate (0.041%) is about 6x Email A's (0.007%), and the difference is statistically significant at the 5% level (p ≈ 0.04). So while the two creatives drive advisors to the site at the same rate, Email B is associated with meaningfully higher DI enrollment within the measurement window.

- Worth flagging to the business: very few of the advisors who enrolled have any web activity tracked in the measurement window at all. Most tracked enrollments for email recipients aren't showing up as a web visit in the window — they may be enrolling by phone, after the window, or through an untracked path. That's a real gap worth raising with the business before treating this web data as the full picture of how advisors enroll.


### Step 8: How many advisors opened and clicked the email?

The business asked for this directly, restricted to advisors who actually received an email (`campaign` 0 or 1).

In [55]:
ab = df_demo_clean[df_demo_clean['campaign'].isin([0, 1])]

summary = ab.groupby('campaign_label').agg(
    sent=('part_id', 'count'),
    bounced=('bounce_dt', 'count'),
    opened=('open_dt', 'count'),
    clicked=('click_dt', 'count'),
)
summary.loc['Total'] = summary.sum()
summary['open_rate'] = summary['opened'] / summary['sent']
summary['click_rate'] = summary['clicked'] / summary['sent']
summary['bounce_rate'] = summary['bounced'] / summary['sent']
summary

,sent,bounced,opened,clicked,open_rate,click_rate,bounce_rate
campaign_label,,,,,,,
Email A,15499,8,1280,60,0.082586,0.003871,0.000516
Email B,21799,23,1856,89,0.085142,0.004083,0.001055
Total,37298,31,3136,149,0.084080,0.003995,0.000831


Roughly **8.4% of sent emails were opened** and **0.4% were clicked** overall, with bounce under 0.1%.

### Step 9: Does randomization produce comparable groups

Before crediting Email B with higher enrollment (Step 7), it's worth checking whether Email A and Email B recipients were actually similar on everything except the creative they got.

For the continuous fields (`age`, `balance`, `tenure`, `fund_ct`, `logon_ct`, `call_ct`) this shows the full distribution (min, 25th/50th/75th percentile, max) alongside the mean, since two groups can have matching averages while still differing in spread or shape; a Welch's t-test on the means backs up whether any visible gap is statistically real. `gender` is categorical, so it's compared with a chi-square test on the group counts instead of a t-test.

In [56]:
email_a_demo = ab[ab['campaign'] == 0]
email_b_demo = ab[ab['campaign'] == 1]

# Continuous fields: full distribution (min/quartiles/max) + t-test on the means

balance_cols = ['age', 'balance', 'tenure', 'fund_ct', 'logon_ct', 'call_ct']
balance_rows = []
for col in balance_cols:
    a_vals = email_a_demo[col].dropna()
    b_vals = email_b_demo[col].dropna()
    t_stat, p_val = stats.ttest_ind(a_vals, b_vals, equal_var=False)
    balance_rows.append({
        'field': col,
        'email_a_mean': a_vals.mean(), 'email_a_min': a_vals.min(), 'email_a_max': a_vals.max(),
        'email_b_mean': b_vals.mean(), 'email_b_min': b_vals.min(), 'email_b_max': b_vals.max(),
        't_stat': t_stat, 'p_value': p_val,
    })

df_balance_check = pd.DataFrame(balance_rows)
df_balance_check

,field,email_a_mean,email_a_min,email_a_max,email_b_mean,email_b_min,email_b_max,t_stat,p_value
0,age,44.995529,42.8,48.40,45.000179,42.8,48.90,-0.483553,0.628706
1,balance,59920.086036,6446.1,117510.57,60018.839861,100.2,126301.31,-0.625007,0.531971
2,tenure,6.015659,2.3,10.00,6.002156,1.7,10.00,1.286307,0.198345
3,fund_ct,6.288470,5.0,11.00,6.293867,5.0,12.00,-0.682715,0.494792
4,logon_ct,12.044971,10.0,17.00,12.040231,10.0,19.00,0.412765,0.679782
5,call_ct,0.057229,0.0,2.00,2.088169,0.0,5.00,-599.878686,0.000000


In [57]:
# Categorical field (gender): counts by group + chi-square test

gender_ct = pd.crosstab(ab['gender_label'], ab['campaign_label'])
chi2_stat, chi2_p, _, _ = stats.chi2_contingency(gender_ct)

print(gender_ct)
print(f"\nchi2 = {chi2_stat:.3f}, p = {chi2_p:.4f}")

campaign_label  Email A  Email B
gender_label                    
Female             4210     5942
Male              11288    15857
Unknown               1        0

chi2 = 1.446, p = 0.4852


Age, balance, tenure, `fund_ct`, and `logon_ct` are all statistically indistinguishable between Email A and Email B (p > 0.19 for all, and their full min/quartile/max ranges overlap closely too), randomization worked fine on those. `gender` is also balanced (chi-square p ≈ 0.49). But `call_ct` is wildly imbalanced. Email A recipients almost all have `call_ct = 0` (94% of them), while Email B recipients almost all have `call_ct = 2` (96% of them).

### Step 10: Combine demographics + web activity + enrollment detail for Tableau

This step joins all three into one denormalized table (rather than just the `ever_enrolled` flag used earlier, the full `enroll_clean` detail — status, enroll/unenroll dates, in-window flag — is brought in here), loads it into `case_study_clean.db`, and exports it as a CSV.


In [58]:
cur.execute("DROP TABLE IF EXISTS advisor_demographics_web")

# TRANSFORM + LOAD: join demographics, web activity, and full enrollment detail

cur.execute("""
    CREATE TABLE advisor_demographics_web AS
    SELECT
        d.part_id, d.fund_ct, d.logon_ct, d.call_ct, d.balance, d.tenure, d.age,
        d.gender, d.gender_label, d.campaign, d.campaign_label,
        d.click_dt, d.bounce_dt, d.open_dt,
        CASE WHEN d.open_dt IS NOT NULL THEN 1 ELSE 0 END AS opened,
        CASE WHEN d.click_dt IS NOT NULL THEN 1 ELSE 0 END AS clicked,
        CASE WHEN d.bounce_dt IS NOT NULL THEN 1 ELSE 0 END AS bounced,
        f.pages_visited, f.reached_welcome,
        f.total_web_hits, f.distinct_pages_visited, f.web_sessions,
        f.first_hit_dt, f.last_hit_dt,
        f.ever_enrolled,
        e.curr_enrll_fl, e.unenrll_fl, e.curr_enrll_status,
        e.enrll_dt, e.unenrll_dt, e.enrolled_in_window
    FROM demo_clean d
    LEFT JOIN advisor_web_summary f ON f.part_id = d.part_id
    LEFT JOIN enroll_clean e ON e.part_id = d.part_id
""")
conn.commit()

df_advisor_demographics_web = pd.read_sql(
    "SELECT * FROM advisor_demographics_web", conn,
    parse_dates=['click_dt', 'bounce_dt', 'open_dt', 'first_hit_dt',
                 'last_hit_dt', 'enrll_dt', 'unenrll_dt']
)
df_advisor_demographics_web.shape


(41767, 31)

Quick check — one row per advisor, demographics and outcomes both present, no rows lost in the join:

In [59]:
print('demo_clean rows:', len(pd.read_sql("SELECT part_id FROM demo_clean", conn)))
print('advisor_demographics_web rows:', len(df_advisor_demographics_web))
print('duplicate part_ids:', df_advisor_demographics_web['part_id'].duplicated().sum())
print('columns:', df_advisor_demographics_web.shape[1])
print()
print('curr_enrll_status among ever-enrolled advisors:')
print(df_advisor_demographics_web.loc[
    df_advisor_demographics_web['ever_enrolled'] == 1, 'curr_enrll_status'
].value_counts())
df_advisor_demographics_web.head()

demo_clean rows: 41767
advisor_demographics_web rows: 41767
duplicate part_ids: 0
columns: 31

curr_enrll_status among ever-enrolled advisors:
curr_enrll_status
Enrolled                   113
No Longer In Plan           22
Unenrolled From Service      8
Name: count, dtype: int64


,part_id,fund_ct,logon_ct,call_ct,balance,tenure,age,gender,gender_label,campaign,...,web_sessions,first_hit_dt,last_hit_dt,ever_enrolled,curr_enrll_fl,unenrll_fl,curr_enrll_status,enrll_dt,unenrll_dt,enrolled_in_window
0,133214,8,11,2,85903.15,5.0,45.1,0,Male,1,...,0,NaT,NaT,0,NaN,NaN,None,NaT,NaT,NaN
1,550202,7,12,2,43173.88,3.3,43.9,0,Male,1,...,0,NaT,NaT,0,NaN,NaN,None,NaT,NaT,NaN
2,577736,6,10,0,43850.56,8.7,45.1,0,Male,0,...,0,NaT,NaT,0,NaN,NaN,None,NaT,NaT,NaN
3,619810,5,13,2,80008.50,7.0,46.3,1,Female,1,...,0,NaT,NaT,0,NaN,NaN,None,NaT,NaT,NaN
4,809111,5,12,2,71166.60,6.6,45.8,0,Male,1,...,0,NaT,NaT,0,NaN,NaN,None,NaT,NaT,NaN


### Step 11: Export to CSV for Tableau

`advisor_demographics_web` is now saved as its own table in `case_study_clean.db` and exported here as a CSV.

In [60]:
df_advisor_demographics_web.to_csv('advisor_demographics_web.csv', index=False)

conn.close()
print('Saved advisor_demographics_web.csv:', df_advisor_demographics_web.shape)

Saved advisor_demographics_web.csv: (41767, 31)
